# Let's build a data agent

The data is the 2026 World Cup: 11 related tables (matches, teams, players, goals, bookings, lineups, substitutions, referees, match_referees, penalty_shootouts, tournaments) in a local DuckDB file.


## Part 1 — Set up the workshop

`smolagents` supplies the agent loop, LiteLLM connects it to Amazon Bedrock, and DuckDB is the warehouse.

In [1]:
%pip install -q "smolagents==1.26.0" "litellm==1.101.0" "boto3==1.43.95" duckdb pandas tabulate ipywidgets


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: /Users/fanisp/Documents/Codex/2026-09-15/smolagents-the-agent-framework-1000-lines/.venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
from pathlib import Path
from bedrock import build_bedrock_model
from notebook_ui import run_agent
import duckdb
from smolagents import tool
from agent import ToolCallingAgent

### Connect to the supplied database

Keep the `.duckdb` file beside this notebook. The fallback path also works in Google Colab after uploading the file to `/content`.

In [3]:
candidates = [Path("worldcups-1930-2026.duckdb"), Path("/content/worldcups-1930-2026.duckdb")]
DB_PATH = next((p for p in candidates if p.exists()), None)
assert DB_PATH is not None, "Place worldcups-1930-2026.duckdb beside the notebook (or upload it to /content in Colab)."

# Read-only at the connection level, not merely in the prompt.
con = duckdb.connect(str(DB_PATH), read_only=True)
TABLES = [row[0] for row in con.execute("SHOW TABLES").fetchall()]

print("database:", DB_PATH.resolve())
print("tables:", TABLES)
print("matches:", con.execute("SELECT count(*) FROM matches").fetchone()[0])

database: /Users/fanisp/Desktop/personal/data-agent-workshop/worldcups-1930-2026.duckdb
tables: ['bookings', 'goals', 'lineups', 'match_referees', 'matches', 'penalty_shootouts', 'players', 'referees', 'substitutions', 'teams', 'tournaments']
matches: 1068


### Configure the model

Press Enter for the default: `claude-haiku-4-5`, deliberately a small model. On simple data a small model fails the way a capable model fails on complex data — valid SQL, wrong interpretation — and it does so 2–3× faster and far cheaper, which matters when the room runs everything live. (Measured: see `stability_haiku_*.json`. A bigger model fails the same questions; comparing models is the bonus at the end.)


In [4]:
model = build_bedrock_model()

Anthropic API key (input hidden; paste and press Enter):  ········
Model [anthropic/claude-haiku-4-5]:  


model: anthropic/claude-haiku-4-5


## Part 2 — A minimal agent (and its first lie)

Two tools only: list the tables, run SQL. No schema documentation, no business rules, no guardrails. First it will genuinely impress you. Then it won't.


In [7]:
MAX_ROWS = 50

@tool
def list_tables() -> str:
    """List the tables available in the World Cup warehouse."""
    return "\n".join(TABLES)

@tool
def execute_sql(query: str) -> str:
    """Call this tool to execute SQL.

    Args:
        query: One SELECT query.
    """
    try:
        df = con.execute(query).df()
    except Exception:
        return "Query failed"
#    except Exception as exc:
#        return f"ERROR: {type(exc).__name__}: {exc}"
    if df.empty:
        return "Query returned 0 rows."
    return df.head(MAX_ROWS).to_markdown(index=False)

In [36]:
MINIMAL_PROMPT = """You are a data analyst answering from a DuckDB warehouse.
Use your tools to answer questions. Keep the final answer concise."""
def make_agent(tools, instructions, max_steps=20, agent_model=None):
    return ToolCallingAgent(
        tools=tools,
        model=agent_model or model,
        instructions=instructions,
        max_steps=max_steps,
    )
minimal_agent = make_agent([list_tables, execute_sql], MINIMAL_PROMPT)

### Try out some questions

- How many matches in the 2026 world cup?
- How many goals have been scored in World Cup history?
- Who won the 2026 World Cup and what was the final score (To get this right, you need to modify the agent a bit)


### Watch the trace, not only the final answer:
- Replay the same questions multiple times. Are they consistent?
- How many steps did it use?
- Did a failed query give it enough information to recover?
- Could a valid SQL give a wrong answer?
- Did it know the schema, or guess column names?


In [10]:
run_agent(minimal_agent, "Who won the 2026 World Cup and what was the final score")

tournament_id,name,season,source_url,license
1,World Cup 1930,1930,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1930/worldcup-full.json,CC0-1.0
2,World Cup 1934,1934,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1934/worldcup-full.json,CC0-1.0
3,World Cup 1938,1938,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1938/worldcup-full.json,CC0-1.0
4,World Cup 1950,1950,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1950/worldcup-full.json,CC0-1.0
5,World Cup 1954,1954,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1954/worldcup-full.json,CC0-1.0
6,World Cup 1958,1958,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1958/worldcup-full.json,CC0-1.0
7,World Cup 1962,1962,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1962/worldcup-full.json,CC0-1.0
8,World Cup 1966,1966,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1966/worldcup-full.json,CC0-1.0
9,World Cup 1970,1970,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1970/worldcup-full.json,CC0-1.0
10,World Cup 1974,1974,https://raw.githubusercontent.com/openfootball/worldcup.json/master/1974/worldcup-full.json,CC0-1.0


## Part 3 — Read the schema first, add a compact data profiler

The agent goes straight into querying, gets errors, realizes it needs to explore the schema. Let's help it by adding a specialized tool to get the schema, nulls, and a few actual values. It avoids dumping entire tables into the context window.
The agent doesn't have to discover it all on it's own through failures.

In [20]:
def _quoted_table(name: str) -> str:
    if name not in TABLES:
        raise ValueError(f"Unknown table {name!r}. Available: {', '.join(TABLES)}")
    return '"' + name.replace('"', '""') + '"'

@tool
def inspect_table(table_name: str) -> str:
    """Profile one table: columns, types, null counts, and example values.

    Args:
        table_name: Exact table name returned by list_tables.
    """
    try:
        table = _quoted_table(table_name)
    except ValueError as exc:
        return f"ERROR: {exc}"
    row_count = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    columns = con.execute(f"DESCRIBE {table}").fetchall()
    lines = [f"table: {table_name}", f"rows: {row_count}", "columns:"]
    for name, dtype, *_ in columns:
        column = '"' + name.replace('"', '""') + '"'
        nulls = con.execute(
            f"SELECT count(*) FILTER (WHERE {column} IS NULL) FROM {table}"
        ).fetchone()[0]
        examples = con.execute(
            f"SELECT DISTINCT {column} FROM {table} WHERE {column} IS NOT NULL LIMIT 4"
        ).fetchall()
        sample = ", ".join(repr(row[0]) for row in examples)
        lines.append(f"- {name}: {dtype}; nulls={nulls}; examples=[{sample}]")
    return "\n".join(lines)

### Try out some questions

Try again the previous question. Should be a smoother flow now.  
- Who won the 2026 World Cup and what was the final score?

Then lets try this one, seems like an easy one. 

- Which player has played the most world cups? 

In [22]:
PROFILE_PROMPT = MINIMAL_PROMPT + """
Before querying, use the inspect table tool to inspect every relevant table. Use observed column names and values.
Resolve internal IDs to human-readable names before answering."""

profile_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    PROFILE_PROMPT,
)


run_agent(profile_agent, "Which player has played the most world cups?")


### A schema profile still does not explain data semantics

The profiler run is *efficient* — a couple of inspect calls, clean queries.. but if you got that Suarez played 7 tournaments. That is not true. He actually played 4. But there have been two other Luis Suarez! The agent gets consused with the schema and messes up. We need to help it a bit more. 


## Part 4 — Add a data contract

Critical semantics are short enough to place directly in the system instructions. The problem that leads to the above problem is a GROUP BY only be name. There are footballers with the same name. But if we group by name + team. Let's see! Note below: 'group appearances by name and team together'

In [38]:
DATA_CONTRACT = """
World Cup warehouse contract

Relationships
- every match belongs to one tournament: matches.tournament_id ->
  tournaments.tournament_id, and tournaments.season is the year (e.g. 2026)
- matches.home_team_id and away_team_id -> teams.team_id
- players.team_id -> teams.team_id
- goals.player_id -> players.player_id
- goals.credited_team_id -> teams.team_id
- lineups, bookings, and penalty_shootouts use match_id, team_id, and player_id
- substitutions use match_id and team_id, with player_on_id and player_off_id
  both pointing to players

Data Guidance
- Player identity and careers
- One players row is one World Cup appearance: one person, one tournament,
  one team. The warehouse stores appearances; it has no table for humans.
- Names are not people. The same name on different national teams is
  different people, and the same name in eras far apart is different
  people. One person keeps one name across all their tournaments - even
  when their country is renamed, a renamed country is still the same team.
 - Career questions count tournaments per person: group appearances by
  name and team together (a name on two different teams is two people),
  and compare people - never names.

Join safety
- goals, bookings, lineups, substitutions, and penalty_shootouts are fact tables.
  Aggregate each fact to the required grain before joining facts together.
"""

CONTEXT_PROMPT = PROFILE_PROMPT + "\n" + DATA_CONTRACT
context_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    CONTEXT_PROMPT,
)

run_agent(context_agent, "which player has played the most world cups?")


- Lionel MESSI and CRISTIANO RONALDO with 6 tournaments is the correct answer.

